# Chapter 13 &mdash; Copy Versus Mirror: the Hierarchy Reversed

**Concept 16 of the Chapter 13 decomposition:** *Copy Versus Mirror: the Hierarchy Reversed*

$w\#w$ needs a tape but copies at a fixed distance; $w\#w^R$ needs only a stack but mirrors at a growing one &mdash; and the window prefers the harder language.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter13-TM/Concept-Copy-Versus-Mirror/Concept-Copy-Versus-Mirror.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Put two languages side by side.

$$L_{copy} = \{\, w\#w \,\}
\qquad\qquad
L_{mirror} = \{\, w\#w^R \,\}$$

By everything this book has built, these are not close. $L_{mirror}$ is
**context-free**: a PDA pushes $w$, then pops it against the reversal, and Chapter 12
would call that an easy afternoon. $L_{copy}$ is **not** context-free &mdash; a stack
hands symbols back in the wrong order &mdash; and needs the tape this chapter spent
its length building.

So the hierarchy says: mirror easy, copy hard.

Now ask a different question, the one a fixed window cares about: **how far back does
the evidence sit?**

* Copy: the symbol at second-half offset $j$ is $|w|+1$ back. A constant.
* Mirror: the symbol at offset $j$ is $2j+2$ back. It **grows as you generate**.

A window of $k$ reaches a constant $|w|+1$ whenever $|w| < k$ &mdash; so the whole
second half is available at once, or none of it is. It reaches a growing $2j+2$ only
while $2j+2 \le k$, so roughly the first $k/2$ symbols come out perfectly and
everything after them is a coin, **whatever $|w|$ is**.

The two languages fail in **different variables**. Measure each against both and the
orderings come out opposite to the hierarchy's. Neither ordering is wrong; they answer
different questions.

## 2. Definitions

### The model, the training loop

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

&nbsp;

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### The corpus, the measurement, one training run

In [ ]:
# --- the corpus.  GENERATED, not filtered --------------------------------
# Strings of the form w#w are exponentially rare, so the enumerate-and-test
# recipe of Chapters 6 and 12 would find almost nothing.  Write the w down
# and double it instead.
#
# `per` words of EVERY length matters more than it looks.  Take all of them
# and the length-6 words outnumber the length-1 words 32 to 1, so every
# per-length number below would be computed from a handful of positions --
# that is how you get a 100% that is one lucky guess.  Balance first.
import random
from itertools import product

SYM = {'0': 0, '1': 1, '#': 2}

def corpus(nmax=6, per=64, seed=0, mirror=False):
    rnd, out = random.Random(seed), []
    for n in range(1, nmax + 1):
        ws = [''.join(w) for w in product('01', repeat=n)]
        for i in range(per):
            w = ws[i % len(ws)]
            out.append(w + '#' + (w[::-1] if mirror else w))
    rnd.shuffle(out)
    return out

def encode(strings):
    """the flat token sequence, plus for each position which second-half
       symbol it is: its |w| and its offset j into the second half."""
    seq, wlen, pos = [], [], []
    for s in strings:
        n = s.index('#')
        for i, c in enumerate(s):
            seq.append(SYM[c])
            wlen.append(n if i > n else None)
            pos.append(i - n - 1 if i > n else None)
    return seq, wlen, pos

# --- accuracy on the second half, grouped by whatever you like -----------
# The model is asked for the single most likely next token (argmax, not a
# sample), and it is asked only at positions that COPY -- the second half.
# 50% is the coin: the two bits are equally likely when you cannot see the
# original.
def accuracy_by(gpt, k, seq, tag):
    idx = [i for i in range(k, len(seq)) if tag[i] is not None]
    hit, tot = {}, {}
    for b in range(0, len(idx), 8192):
        chunk = idx[b:b + 8192]
        X = torch.tensor([seq[i - k:i] for i in chunk], dtype=torch.long)
        pred = gpt(X).argmax(-1).tolist()
        for i, p in zip(chunk, pred):
            t = tag[i]
            tot[t] = tot.get(t, 0) + 1
            hit[t] = hit.get(t, 0) + (p == seq[i])
    return {t: (hit[t] / tot[t], tot[t]) for t in sorted(tot)}

def table(rows, keys, label):
    print('%-9s' % label + ''.join('%7s' % k for k in keys))
    for name, acc in rows:
        print('%-9s' % name + ''.join('%6.0f%%' % (100 * acc[k][0])
                                      for k in keys))
    print('%-9s' % 'samples' + ''.join('%7d' % rows[0][1][k][1] for k in keys))

# --- one training run, with every knob in the signature ------------------
def train_at(seq, k, iters=300, n_embd=16, seed=1337, quiet=False):
    X, Y = make_XY(seq, k)
    config = GPTConfig(block_size=k, vocab_size=3, n_layer=4, n_head=4,
                       n_embd=n_embd, bias=False)
    torch.manual_seed(seed)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters,
                       every=iters if quiet else iters // 4)
    return gpt, losses[-1]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;15.&nbsp;The Context Window Is the Memory](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter13-TM/Concept-Context-Window-Is-The-Memory/Concept-Context-Window-Is-The-Memory.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13-TM/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;1.&nbsp;Why Study Impossibility Results?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter14-Interp/Concept-Why-Impossibility-Results/Concept-Why-Impossibility-Results.ipynb)&nbsp;&rarr;

---

## 3. Tests

The mirror language is context-free, and here is the PDA that proves it. One Jove detail: `#` is a PDA's bottom-of-stack marker and may not also be an input symbol, so the machine below spells the separator `c` &mdash; which is how the textbooks write this language anyway. `Z` marks the bottom of the pushed word.

In [ ]:
MIRROR = md2mc('''PDA
I  : 0 , # ; 0Z# -> I      !! first symbol: push it, and mark the word bottom
I  : 1 , # ; 1Z# -> I
I  : 0 , 0 ; 00 -> I       !! the rest of w: push each symbol
I  : 0 , 1 ; 01 -> I
I  : 1 , 0 ; 10 -> I
I  : 1 , 1 ; 11 -> I
I  : c , 0 ; 0 -> M        !! cross the separator, keeping the stack
I  : c , 1 ; 1 -> M
I  : c , # ; # -> F        !! the empty word: the input is just "c"
M  : 0 , 0 ; '' -> M       !! pop each symbol against the reversal
M  : 1 , 1 ; '' -> M
M  : '' , Z ; '' -> F      !! the word is used up exactly
''')

def pda_accepts(P, s, STKMAX=12):
    return len(run_pda(s, P, acceptance='ACCEPT_F',
                       STKMAX=STKMAX, chatty=False)[1]) > 0

from itertools import product
cases = [a + '#' + b for n in range(5) for a in map(''.join, product('01', repeat=n))
                     for m in range(5) for b in map(''.join, product('01', repeat=m))]
bad = [s for s in cases
       if pda_accepts(MIRROR, s.replace('#', 'c'))
          != (s.split('#')[0][::-1] == s.split('#')[1])]
print('checked %d strings; the PDA disagrees on : %s' % (len(cases), bad))
assert not bad
print('a STACK decides the mirror language -- no tape needed')

A stack cannot do the copy language, which is why Concept 9 needed a machine with a tape. Two corpora, same recipe, one flag apart.

In [ ]:
copy_s = corpus(nmax=6, per=64, mirror=False)
mirr_s = corpus(nmax=6, per=64, mirror=True)
print('copy   :', copy_s[0], ' ', copy_s[1])
print('mirror :', mirr_s[0], ' ', mirr_s[1])
seqC, wlenC, posC = encode(copy_s)
seqM, wlenM, posM = encode(mirr_s)

**Train both, at the same window.** $k=6$, so the prediction is: copy is floored from $|w|=6$; mirror is floored from $j=2$, since $2j+2 > 6$ once $j \ge 3$.

In [ ]:
k = 6
gptC, flC = train_at(seqC, k, iters=1500, quiet=True)
gptM, flM = train_at(seqM, k, iters=1500, quiet=True)
print('copy   final loss %.4f' % flC)
print('mirror final loss %.4f' % flM)

**Grouped by $|w|$** &mdash; the variable that governs the copy language.

In [ ]:
table([('copy', accuracy_by(gptC, k, seqC, wlenC)),
       ('mirror', accuracy_by(gptM, k, seqM, wlenM))],
      list(range(1, 7)), '|w| =')

**Grouped by $j$**, the offset into the second half &mdash; the variable that governs the mirror language.

In [ ]:
table([('copy', accuracy_by(gptC, k, seqC, posC)),
       ('mirror', accuracy_by(gptM, k, seqM, posM))],
      list(range(0, 6)), 'j =')

Read the second table.

In [ ]:
print("Mirror, by j: the first few offsets come out at or near 100%, and")
print("then it is 50% for the rest -- a wall at exactly the j where 2j+2")
print("passes k=6.  Grouped by |w| instead, the same model merely looks")
print("like it is degrading gently, because longer words simply contain")
print("more of the high-j positions.  Same model, same numbers, and one")
print("grouping tells you what is happening while the other hides it.")
print()
print("Copy is the other way round: sharp in |w|, blurred in j.")
print()
print("Choosing the variable to group by IS the experiment.")

And the point worth taking to Chapter 14.

In [ ]:
print("The Chomsky hierarchy ranks these two languages:")
print("    mirror  context-free      a PDA does it")
print("    copy    not context-free  it needs a tape")
print()
print("The context window ranks them by how far back the evidence sits:")
print("    mirror  distance 2j+2, GROWS     -- dark after about k/2 symbols")
print("    copy    distance |w|+1, CONSTANT -- fine whenever |w| < k")
print()
print("Opposite orders.  That is not a paradox and the hierarchy is not")
print("wrong: it ranks languages by the MEMORY a machine needs, and a")
print("context window is not a memory -- it is a fixed-length view.  What")
print("a view cares about is distance; what a machine cares about is")
print("structure.  Two questions, two orderings, both correct.")

## 4. Exercises


1. Predict the mirror wall for $k=8$ and $k=10$ from $2j+2 \le k$, then measure.
2. Group the mirror model by $|w|$ only, and write the conclusion a careless reader
   would draw. Then say what is wrong with it.
3. $w\#w^R$ with $|w|$ large is a palindrome problem. Explain why reading a palindrome
   *backwards from the end* would make it a constant-distance job, and what that says
   about generation order.
4. Build the third member of the family, $\{w\#w\#w\}$. Where does it sit in the
   hierarchy, and what is the copy distance?
5. `MIRROR` above accepts by final state. Rewrite it to accept by empty stack and check
   it agrees, using Concept 5 of Chapter 12.
6. One sentence, for someone who has not read this chapter: why does a language being
   *higher* in the Chomsky hierarchy not make it harder for a transformer?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13-TM/Concept-Copy-Versus-Mirror')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')